---
title: "Decision Trees and Random Forests"
execute:
  enabled: true
jupyter: python3
---

You're building a pricing tool for Airbnb hosts. In [Chapter 5](lec05-feature-engineering.qmd), feature engineering got you to a reasonable R² — but you had to choose which interactions to include, which polynomial degree, and how to encode dozens of neighborhoods. Every choice required judgment.

What if the model could figure out where to split on its own? Decision trees do exactly that: they find the splits automatically. No encoding, no polynomial terms, no missing-value imputation. The model looks at the data and decides where to cut.

:::{.callout-tip}
## Two model families × two tasks

In Chapters 4--6, you used **linear models** for regression (predicting prices). In [Chapter 7](lec07-classification.qmd), you used a linear model for classification (logistic regression). Today introduces a completely different model family --- **trees** --- that handles both tasks:

|              | Regression | Classification |
|--------------|-----------|----------------|
| **Linear models** | Linear regression (Ch 4--5) | Logistic regression (Ch 7) |
| **Trees** | Regression tree (today) | Classification tree (today) |

Logistic regression is much more complex to optimize than linear regression (gradient descent vs. closed-form normal equations). For trees, regression and classification are equally simple --- just change the splitting criterion.
:::

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (r2_score, mean_squared_error, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             roc_curve, roc_auc_score, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

## The data: Airbnb NYC listings

In [ ]:
DATA_DIR = 'data'
listings = pd.read_csv(f'{DATA_DIR}/airbnb/listings.csv', low_memory=False)

cols = ['id', 'name', 'description', 'price', 'bedrooms', 'bathrooms', 'room_type',
        'neighbourhood_group_cleansed', 'accommodates', 'number_of_reviews',
        'latitude', 'longitude']
df = listings[cols].dropna(subset=['price', 'bedrooms', 'bathrooms']).copy()
df = df.rename(columns={'neighbourhood_group_cleansed': 'borough'})

# Convert price to numeric
df['price'] = pd.to_numeric(df['price'].astype(str).str.replace('[$,]', '', regex=True), errors='coerce')

# Focus on reasonable prices (filter out near-free and luxury outliers)
df = df[df['price'].between(10, 500)].reset_index(drop=True)
prices = df['price']

print(f"Working with {len(df):,} listings")
df[['name', 'price', 'bedrooms', 'bathrooms', 'room_type', 'borough']].head()

We prepare the same feature matrix used in [Chapter 5](lec05-feature-engineering.qmd): numeric features plus one-hot encoded categoricals via `pd.get_dummies()`.

In [ ]:
# One-hot encode categorical features; include continuous features
# (latitude, longitude, accommodates, number_of_reviews) so trees have
# real numeric signal to split on
cat_features = pd.get_dummies(df[['room_type', 'borough']], drop_first=True)
num_features = df[['bedrooms', 'bathrooms', 'accommodates', 'number_of_reviews',
                   'latitude', 'longitude']]
X_base = pd.concat([num_features, cat_features], axis=1)
feature_names = list(X_base.columns)

print(f"Feature matrix: {X_base.shape[0]} rows × {X_base.shape[1]} columns")
print("Features:", feature_names)

## A different approach — decision trees

Engineering features by hand is creative but tedious. What if the model could find the right splits on its own?

A **decision tree** splits the feature space into regions and predicts the mean price in each region. Each internal node asks an if/then question about one feature: "Is bedrooms > 2?" or "Is borough = Manhattan?" The tree learns which questions to ask and where to split — no manual feature engineering needed.

We hold out 30% of listings as a test set so we can measure honest generalization, following the train/test split from [Chapter 6](lec06-validation.qmd).

In [ ]:
# Hold out 30% of listings for honest evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X_base, prices, test_size=0.3, random_state=42
)

# Decision tree with limited depth to prevent overfitting
tree = DecisionTreeRegressor(max_depth=4, random_state=42)
tree.fit(X_train, y_train)

tree_train_r2 = tree.score(X_train, y_train)
tree_test_r2 = tree.score(X_test, y_test)

print(f"Decision Tree (depth=4):")
print(f"  Train R² = {tree_train_r2:.3f}  |  Test R² = {tree_test_r2:.3f}")

Let's visualize what the tree learned using `plot_tree`.

In [ ]:
# Visualize the tree (using max_depth=3 for readability)
tree_viz = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_viz.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(tree_viz, feature_names=feature_names, filled=True, fontsize=8,
          rounded=True, ax=ax, impurity=False, proportion=False)
ax.set_title('Decision tree for Airbnb price prediction (depth=3)', fontsize=14)
plt.tight_layout()
plt.show()

Each leaf shows the predicted price (the mean of training examples that land there) and how many samples it contains. The tree automatically discovers that room type and borough matter — no one-hot encoding intuition required.

:::{.callout-tip}
## Think About It
The tree picks its own split points. Look at the first split — does it match your intuition about what matters most for price?
:::


### How a decision tree splits

A decision tree partitions the feature space using a **greedy** algorithm (**greedy**: chooses the locally best split at each step without planning ahead). At each node, it considers every feature and every possible threshold, choosing the split that produces the most homogeneous child nodes.

For regression, "homogeneous" means low variance. The algorithm picks the split that minimizes the total squared error in the two child nodes:

$$\min_{j,\, s} \left[ \sum_{x_i \in R_1(j,s)} (y_i - \bar{y}_1)^2 + \sum_{x_i \in R_2(j,s)} (y_i - \bar{y}_2)^2 \right]$$

where $R_1(j,s) = \{x : x_j \le s\}$ and $R_2(j,s) = \{x : x_j > s\}$ are the two regions created by splitting feature $j$ at threshold $s$, and $\bar{y}_1, \bar{y}_2$ are the mean outcomes in each region. Each leaf predicts the mean of the training observations that fall into it.

For classification, the criterion is **Gini impurity** instead of squared error. A node is "pure" if every observation in it has the same label. The Gini index $\hat{p}(1 - \hat{p})$ measures how mixed a node is, where $\hat{p}$ is the fraction of observations with label 1 (for binary classification; generalizes to $1 - \sum_k \hat{p}_k^2$ for multi-class). The algorithm greedily picks splits that maximize the total decrease in impurity.

The process is **recursive**: after the first split creates two child nodes, the algorithm splits each child independently, and so on. The tree grows until a stopping criterion is met — typically a maximum depth or a minimum number of samples per leaf.

In [ ]:
# Text representation of the tree structure
print(export_text(tree_viz, feature_names=feature_names, max_depth=3))

## Overfitting a single tree

A decision tree with no depth limit will keep splitting until every leaf contains a single training observation. The result: perfect training accuracy and terrible generalization.

In [ ]:
# Fit trees at increasing depths and track train vs test R²
depths = [1, 2, 3, 4, 5, 7, 10, 15, 20, None]
depth_labels = [str(d) if d else 'None' for d in depths]
train_r2s = []
test_r2s = []

for d in depths:
    t = DecisionTreeRegressor(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_r2s.append(t.score(X_train, y_train))
    test_r2s.append(t.score(X_test, y_test))
    label = f"depth={str(d):>4s}"
    print(f"{label}:  Train R² = {train_r2s[-1]:.3f}  |  Test R² = {test_r2s[-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = range(len(depths))
ax.plot(x_pos, train_r2s, 'o-', color='steelblue', linewidth=2, markersize=8, label='Train R²')
ax.plot(x_pos, test_r2s, 's-', color='orangered', linewidth=2, markersize=8, label='Test R²')
ax.set_xticks(x_pos)
ax.set_xticklabels(depth_labels)
ax.set_xlabel('Tree depth (max_depth)')
ax.set_ylabel('R²')
ax.set_title('A single tree overfits as depth increases')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Training R² climbs from 0.30 toward 1.00 while test R² peaks around depth 7 and then collapses.

The pattern is the **bias-variance tradeoff** from [Chapter 6](lec06-validation.qmd) in action. A shallow tree has high bias (it underfits — too few splits to capture the real pattern) but low variance (it's stable across different training samples). A deep tree has low bias but high variance — it memorizes the noise in whatever training data it happens to see. The sweet spot balances these two forces. In Chapter 6 we saw this tradeoff with polynomial degree; here the "complexity knob" is tree depth. Extreme shallow trees have very low variance; moderate-depth trees already show meaningful sensitivity to the training set.

Let's visualize the predictions from a shallow and deep tree to see what overfitting looks like.

In [ ]:
# Compare shallow vs deep tree predictions along the bedrooms axis
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Create prediction grid: vary bedrooms, hold other features at median
median_features = X_train.median()
bedroom_range = np.arange(0, 7, 0.1)
X_slice = pd.DataFrame(
    np.tile(median_features.values, (len(bedroom_range), 1)),
    columns=X_train.columns
)
X_slice['bedrooms'] = bedroom_range

for ax, depth, title in zip(axes,
                             [3, 20],
                             ['Shallow tree (depth=3)', 'Deep tree (depth=20)']):
    t = DecisionTreeRegressor(max_depth=depth, random_state=42)
    t.fit(X_train, y_train)
    preds = t.predict(X_slice)

    ax.scatter(df['bedrooms'] + np.random.normal(0, 0.05, len(df)),
               prices, alpha=0.02, s=3, color='gray', label='data')
    ax.plot(bedroom_range, preds, color='orangered', linewidth=2.5, label=f'depth={depth}')
    ax.set_xlabel('Bedrooms')
    ax.set_ylabel('Predicted price ($)')
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(-0.5, 6.5)

plt.tight_layout()
plt.show()

The shallow tree produces smooth, interpretable predictions. The deep tree produces jagged step functions that jump wildly between adjacent bedroom counts — it has memorized the training data's quirks rather than learning the underlying pattern.

:::{.callout-tip}
## Think About It
A deep tree achieves a training R² close to 1.0. Why doesn't that make it a good model? What would you look at to decide when to stop growing the tree?
:::


## Choosing tree depth by cross-validation

The depth experiment above shows that deeper isn't always better. How do we pick the right depth? The same tool from [Chapter 6](lec06-validation.qmd): **cross-validation**. We fit a decision tree at each candidate depth, score it with 5-fold CV, and pick the depth with the best average test performance.

You might ask: the depth experiment above already showed test R² peaking around depth 7. Why do we need CV? Two reasons. First, a single train/test split is noisy; CV averages over 5 splits for a stabler estimate. Second — more importantly — if you tune the depth by peeking at the test set, the test set is no longer a clean held-out estimate of deployment performance. Cross-validation does the tuning on the training data, keeping the test set untouched for a final honest report.

In [ ]:
depths_cv = [1, 2, 3, 4, 5, 7, 10, 15, 20]
cv_means = []
cv_stds = []

for d in depths_cv:
    tree_cv = DecisionTreeRegressor(max_depth=d, random_state=42)
    scores = cross_val_score(tree_cv, X_train, y_train, cv=5, scoring='r2')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    print(f"depth={d:>2d}: CV R² = {scores.mean():.4f} ± {scores.std():.4f}")

best_depth = depths_cv[np.argmax(cv_means)]
print(f"\nBest depth by CV: {best_depth}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(depths_cv, cv_means, yerr=cv_stds, fmt='o-', color='steelblue',
            linewidth=2, markersize=8, capsize=5, label='CV R² ± 1 std')
ax.axvline(best_depth, color='orangered', linestyle='--', linewidth=2,
           label=f'Best depth = {best_depth}')
ax.set_xlabel('Tree depth (max_depth)')
ax.set_ylabel('Cross-validated R²')
ax.set_title('Choosing tree depth by cross-validation')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The U-shaped CV curve mirrors the polynomial experiment from [Chapter 6](lec06-validation.qmd): too shallow underfits, too deep overfits, and cross-validation finds the sweet spot.

:::{.callout-tip}
## Same framework, different model

In [Chapter 6](lec06-validation.qmd), you used 5-fold cross-validation to pick the best polynomial degree. Here you just used the same workflow — `cross_val_score` with a 5-fold CV — to pick the best tree depth. **The same framework generalizes to any hyperparameter.** Lasso $\alpha$, polynomial degree, tree depth, number of neighbors, regularization strength — all are picked by the same mechanical procedure: sweep candidate values, score each by CV, take the best.
:::

:::{.callout-tip}
## Think About It
Compare this CV curve to the polynomial degree CV curve from Chapter 6. Both show a peak followed by decline. What's the "complexity knob" in each case, and why does test performance decline past the peak?
:::

Let's refit a tree at the best CV depth and evaluate on the held-out test set for an honest final score.

In [ ]:
best_tree = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
best_tree.fit(X_train, y_train)
print(f"Tree at CV-best depth={best_depth}: Test R² = {best_tree.score(X_test, y_test):.3f}")

Cross-validation picked the depth without ever touching the test set. The test-set score reported above is an honest estimate of how this tree will perform on new listings.


## Classification trees

Everything above predicts a continuous price. But trees handle **classification** just as naturally. The splitting criterion changes from squared error to **Gini impurity**, but everything else --- greedy recursive partitioning, depth control, cross-validation --- stays the same.

Let's create a binary target: is a listing "expensive" (price above $200)?

In [ ]:
# Binary target: expensive = price > $200
y_train_exp = (y_train > 200).astype(int)
y_test_exp = (y_test > 200).astype(int)

print(f"Expensive listings: {y_train_exp.mean():.1%} of training set, "
      f"{y_test_exp.mean():.1%} of test set")

We pick the tree depth by cross-validation, just as we did for regression.

In [ ]:
depths_clf = [1, 2, 3, 4, 5, 7, 10, 15, 20]
cv_means_clf = []

for d in depths_clf:
    clf_cv = DecisionTreeClassifier(max_depth=d, random_state=42)
    scores = cross_val_score(clf_cv, X_train, y_train_exp, cv=5, scoring='accuracy')
    cv_means_clf.append(scores.mean())
    print(f"depth={d:>2d}: CV accuracy = {scores.mean():.4f} ± {scores.std():.4f}")

best_depth_clf = depths_clf[np.argmax(cv_means_clf)]
print(f"\nBest depth by CV: {best_depth_clf}")

In [ ]:
# Fit classification tree at best depth
clf_tree = DecisionTreeClassifier(max_depth=best_depth_clf, random_state=42)
clf_tree.fit(X_train, y_train_exp)

y_pred_tree = clf_tree.predict(X_test)
y_prob_tree = clf_tree.predict_proba(X_test)[:, 1]

print(f"Classification tree (depth={best_depth_clf}):")
print(f"  Accuracy  = {clf_tree.score(X_test, y_test_exp):.3f}")
print(f"  Precision = {precision_score(y_test_exp, y_pred_tree):.3f}")
print(f"  Recall    = {recall_score(y_test_exp, y_pred_tree):.3f}")
print(f"  F1        = {f1_score(y_test_exp, y_pred_tree):.3f}")
print(f"  AUC       = {roc_auc_score(y_test_exp, y_prob_tree):.3f}")

The confusion matrix shows how the tree's predictions break down.

In [ ]:
#| code-fold: true
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test_exp, y_pred_tree)
disp = ConfusionMatrixDisplay(cm, display_labels=['< $200', '≥ $200'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Classification tree (depth={best_depth_clf})')
plt.tight_layout()
plt.show()

The ROC curve plots the tradeoff between true positive rate (recall) and false positive rate as we vary the classification threshold.

In [ ]:
#| code-fold: true
fpr_tree, tpr_tree, _ = roc_curve(y_test_exp, y_prob_tree)
auc_tree = roc_auc_score(y_test_exp, y_prob_tree)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_tree, tpr_tree, color='steelblue', linewidth=2.5,
        label=f'Classification tree (AUC = {auc_tree:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random guess')
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate (recall)')
ax.set_title('ROC curve — classification tree')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The classification tree learns the same kinds of splits as the regression tree — room type, borough, bedrooms — but now it routes listings to "expensive" or "not expensive" leaves instead of predicting a dollar amount. If you already understand regression trees, you understand classification trees.


## Random forests

One tree overfits. What about averaging many trees?

One tree overfits. What about averaging many trees?

A **random forest** fits hundreds of decision trees, each on a random subset of the data and features, then averages their predictions. The randomness ensures the trees disagree on noise but agree on signal.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_train_r2 = rf.score(X_train, y_train)
rf_test_r2 = rf.score(X_test, y_test)

print("Model comparison (test R²):")
print(f"  Decision Tree (depth=4):     {tree_test_r2:.3f}")
print(f"  Random Forest (100 trees):   {rf_test_r2:.3f}")

The forest beats the single tree — with no hyperparameter tuning beyond the tree count.


### How random forests work: bagging + feature subsampling

A random forest combines two ideas:

1. **Bagging** (bootstrap aggregating): each tree trains on a different bootstrap sample — a random draw (with replacement) from the training data. Each sample is the same size as the original dataset but contains some observations multiple times and omits others. For example, if the original dataset has 3 observations $(A, B, C)$, a bootstrap sample might be $(A, B, B)$ or $(C, C, A)$ — same size, random draws with replacement.

2. **Feature subsampling**: at each split, the tree considers only a random subset of features (typically $\sqrt{p}$ out of $p$ total features). This forces the trees to find different splits, reducing the correlation between trees.

The ensemble prediction is the average of all individual tree predictions (for regression) or majority vote (for classification).

Each individual tree is overfit — it has memorized the noise in its particular bootstrap sample. Different trees memorize *different* noise, so averaging cancels it.


### More trees never hurts

Let's train random forests with increasing numbers of trees and track test performance.

In [ ]:
# Build feature matrix for the full train/test sets
n_trees_list = [1, 5, 10, 25, 50, 100, 200, 500]
rf_test_scores = []

for n_trees in n_trees_list:
    rf_temp = RandomForestRegressor(n_estimators=n_trees, random_state=42, n_jobs=-1)
    rf_temp.fit(X_train, y_train)
    score = rf_temp.score(X_test, y_test)
    rf_test_scores.append(score)
    print(f"n_estimators = {n_trees:>3d}: Test R² = {score:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(n_trees_list, rf_test_scores, 'o-', color='forestgreen', linewidth=2.5, markersize=10)
ax.set_xlabel('Number of trees (n_estimators)')
ax.set_ylabel('Test R²')
ax.set_title('Random forest: more trees = better or flat — never worse')
ax.set_xscale('log')
ax.set_xticks(n_trees_list)
ax.set_xticklabels(n_trees_list)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

No U-shape! The test R² increases and then plateaus. Adding more trees never hurts — it just stops helping. The pattern is fundamentally different from the polynomial and tree-depth experiments in [Chapter 6](lec06-validation.qmd), where more complexity eventually *destroyed* test performance. (Caveat: this "more is never worse" property is specific to bagging/random forests. For *gradient boosting*, which we'll meet in [Chapter 17](lec17-automl-llms.qmd), adding too many trees *can* overfit.)

:::{.callout-note}
## Escaping the classical U-curve

This is the "one known escape" that [Chapter 6](lec06-validation.qmd) promised in its *Beyond the classical tradeoff* callout. The classical bias-variance curve says test error is U-shaped in model complexity: simple models underfit, complex models overfit, and you have to choose the sweet spot by cross-validation. That story assumes you are fitting a *single* model. The forest is not a single model — it is an average over hundreds of them. Averaging changes the rules: as long as individual trees' errors are sufficiently uncorrelated, adding more trees drives variance down without pushing bias up. There is no sweet spot to find on the `n_estimators` axis — "more" is monotonically at-least-as-good. The U-curve you saw for polynomial degree in Chapter 6 has been replaced by a one-sided staircase.
:::


### Individual trees are overfit; their average is smooth

To understand why averaging works, let's look at what individual trees actually predict. We'll train 5 decision trees on bootstrap samples and plot their predictions along a 1D slice: price vs. bedrooms, holding other features at their median values.

In [ ]:
# Create a 1D prediction slice: vary bedrooms, hold everything else at median
median_features = X_train.median()

bedroom_range = np.arange(0, 7, 0.1)
X_slice = pd.DataFrame(
    np.tile(median_features.values, (len(bedroom_range), 1)),
    columns=X_train.columns
)
X_slice['bedrooms'] = bedroom_range

# Train 5 individual trees on bootstrap samples
np.random.seed(42)
fig, ax = plt.subplots(figsize=(10, 6))

individual_preds = []
for i in range(5):
    # Bootstrap sample
    boot_idx = np.random.choice(len(X_train), size=len(X_train), replace=True)
    X_boot = X_train.iloc[boot_idx]
    y_boot = y_train.iloc[boot_idx]

    tree_i = DecisionTreeRegressor(max_depth=None, random_state=i)
    tree_i.fit(X_boot, y_boot)
    preds = tree_i.predict(X_slice)
    individual_preds.append(preds)
    ax.plot(bedroom_range, preds, alpha=0.4, linewidth=1.2, label=f'Tree {i+1}')

# Average prediction
avg_preds = np.mean(individual_preds, axis=0)
ax.plot(bedroom_range, avg_preds, color='black', linewidth=3, label='Average (5 trees)', zorder=5)

ax.set_xlabel('Bedrooms')
ax.set_ylabel('Predicted price ($)')
ax.set_title('Each tree is overfit and wiggly. Their average is smooth.')
ax.legend(fontsize=10, loc='upper left')
ax.set_xlim(0, 6)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Each individual tree (thin colored lines) is wiggly and overfit — it has memorized the quirks of its particular bootstrap sample. But their *errors point in different directions*. One tree might predict too high for 3-bedroom listings while another predicts too low. When we average them, the noise cancels out, and what remains is the genuine signal.

The core insight of **bagging** (bootstrap aggregating): averaging many overfit models reduces variance without increasing bias. It's the same principle behind asking 100 people to estimate the weight of a cow at a county fair — individual guesses vary wildly, but the average is remarkably accurate.

:::{.callout-note}
## The bias-variance decomposition, revisited

[Chapter 6](lec06-validation.qmd) wrote expected test error as $\text{bias}^2 + \text{variance} + \sigma^2$. A single deep tree is a *low-bias, high-variance* estimator: given enough depth, it can approximate any function (low bias), but it is exquisitely sensitive to the particular training sample (high variance). Regularizing a single tree — pruning it, capping its depth — lowers variance at the cost of raising bias, and that is the tradeoff Chapter 6 taught us to navigate with cross-validation.

A random forest does something structurally different. It keeps every tree deep (bias stays low) and reduces variance by *averaging*. If the individual trees had perfectly correlated errors, averaging would do nothing. The bootstrap sampling and feature subsampling are there precisely to de-correlate their mistakes, so that averaging can work. The net effect: the forest inherits the low bias of a deep tree and the low variance of a stable estimator, without paying bias to get there. This is why random forests are a go-to benchmark — they sidestep the tuning dance that individual models require, and they do it by engineering a favorable bias-variance decomposition, not by finding a sweet spot on a U-curve.
:::


## Random forest as benchmark

How does the random forest compare to the hand-engineered linear model from [Chapter 5](lec05-feature-engineering.qmd)? Compare their test R².

In [ ]:
# Linear model with interactions (same as Chapter 5)
borough_dummies = pd.get_dummies(df['borough'], drop_first=True)
interactions = borough_dummies.multiply(df['bedrooms'], axis=0)
interactions.columns = [f'bedrooms_x_{col}' for col in interactions.columns]
X_interact = pd.concat([X_base, interactions], axis=1)

lm_interact = LinearRegression().fit(
    X_interact.loc[X_train.index], y_train
)
lm_test_r2 = r2_score(y_test, lm_interact.predict(X_interact.loc[X_test.index]))

# Random forest (already fit above)
print("Test R² comparison:")
print(f"  Linear model (bedrooms + bath + room + borough):         {r2_score(y_test, LinearRegression().fit(X_train, y_train).predict(X_test)):.3f}")
print(f"  Linear model + interactions:                             {lm_test_r2:.3f}")
print(f"  Decision tree (depth=4):                                 {tree_test_r2:.3f}")
print(f"  Random forest (100 trees):                               {rf_test_r2:.3f}")

The random forest wins — and required zero feature engineering. No polynomial terms, no interaction terms, no decisions about which features to cross. For tabular prediction tasks, a random forest is a strong default benchmark: if your hand-crafted model can't beat it, the engineering effort may not be worthwhile.

:::{.callout-tip}
## Think About It
The random forest beats the linear model with interactions. Does that mean linear models are useless? What advantages might a linear model still have? (Hint: think about interpretability and coefficient meaning.)
:::


### Random forest classifier

Same story for classification. A random forest classifier fits many classification trees on bootstrap samples and takes a majority vote.

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train_exp)

y_pred_rf = rf_clf.predict(X_test)
y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]

print(f"Random forest classifier (100 trees):")
print(f"  Accuracy  = {rf_clf.score(X_test, y_test_exp):.3f}")
print(f"  Precision = {precision_score(y_test_exp, y_pred_rf):.3f}")
print(f"  Recall    = {recall_score(y_test_exp, y_pred_rf):.3f}")
print(f"  F1        = {f1_score(y_test_exp, y_pred_rf):.3f}")
print(f"  AUC       = {roc_auc_score(y_test_exp, y_prob_rf):.3f}")

In [ ]:
#| code-fold: true
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm_rf = confusion_matrix(y_test_exp, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=['< $200', '≥ $200'])
disp_rf.plot(ax=axes[0], cmap='Greens', values_format='d')
axes[0].set_title('Random forest classifier (100 trees)')

# ROC comparison: single tree vs forest
fpr_rf, tpr_rf, _ = roc_curve(y_test_exp, y_prob_rf)
auc_rf = roc_auc_score(y_test_exp, y_prob_rf)

axes[1].plot(fpr_tree, tpr_tree, color='steelblue', linewidth=2,
             label=f'Single tree (AUC = {auc_tree:.3f})')
axes[1].plot(fpr_rf, tpr_rf, color='forestgreen', linewidth=2.5,
             label=f'Random forest (AUC = {auc_rf:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random guess')
axes[1].set_xlabel('False positive rate')
axes[1].set_ylabel('True positive rate (recall)')
axes[1].set_title('ROC: single tree vs. random forest')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Same story as regression: averaging reduces variance. The forest's ROC curve sits above the single tree's at nearly every threshold, and its AUC is higher. The forest gets more of the expensive listings right without flagging too many cheap ones.


## Trees vs. logistic regression

We now have two classification approaches for the same task: trees and logistic regression from [Chapter 7](lec07-classification.qmd). How do they compare?

In [ ]:
# Logistic regression on the same binary target
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train_exp)

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_test_exp, y_prob_lr)
auc_lr = roc_auc_score(y_test_exp, y_prob_lr)

In [ ]:
# Side-by-side comparison
print(f"{'Metric':<12} {'Logistic Reg':>14} {'Tree (d={})'.format(best_depth_clf):>14} {'RF (100)':>14}")
print("-" * 58)
for name, fn in [('Precision', precision_score), ('Recall', recall_score),
                 ('F1', f1_score)]:
    print(f"{name:<12} {fn(y_test_exp, y_pred_lr):>14.3f} "
          f"{fn(y_test_exp, y_pred_tree):>14.3f} "
          f"{fn(y_test_exp, y_pred_rf):>14.3f}")
print(f"{'AUC':<12} {auc_lr:>14.3f} {auc_tree:>14.3f} {auc_rf:>14.3f}")

In [ ]:
#| code-fold: true
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_lr, tpr_lr, color='darkorange', linewidth=2.5,
        label=f'Logistic regression (AUC = {auc_lr:.3f})')
ax.plot(fpr_tree, tpr_tree, color='steelblue', linewidth=2,
        label=f'Classification tree (AUC = {auc_tree:.3f})')
ax.plot(fpr_rf, tpr_rf, color='forestgreen', linewidth=2.5,
        label=f'Random forest (AUC = {auc_rf:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random guess')
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate (recall)')
ax.set_title('ROC comparison: logistic regression vs. trees')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Each approach has distinct strengths:

- **Logistic regression** produces interpretable coefficients (log-odds per feature). You can say "each additional bedroom increases the odds of being expensive by a factor of $e^{\beta}$." The model is fast, stable, and works well when the true decision boundary is roughly linear.

- **Decision trees** capture nonlinear relationships and interactions without manual feature engineering. A tree can learn that "Manhattan listings with 2+ bedrooms are expensive" without you specifying the interaction. The tree structure is easy to visualize and explain to non-technical stakeholders.

- **Random forests** combine the nonlinear power of trees with the variance reduction of ensembling. They're harder to interpret than either single model, but they consistently perform well on tabular data with minimal tuning.

When to prefer each: use logistic regression when you need interpretable coefficients or when the dataset is small and you want a stable model. Use trees or forests when you have many features with potential interactions and don't need per-feature coefficient interpretations.

:::{.callout-tip}
## The 2×2 matrix, completed

We started this chapter with a 2×2 table: linear models and trees, each applied to regression and classification. You've now seen all four cells. The key lesson: model families (linear vs. tree-based) and tasks (regression vs. classification) are independent choices. Once you learn a model family, applying it to either task is straightforward.
:::


## Feature importance

A random forest is harder to interpret than a linear regression — you can't point to a single coefficient and say "each bedroom adds \$X." But the forest does tell you which features it relies on most.

**Feature importance** measures how much each feature contributes to reducing prediction error across all trees in the forest. Specifically, scikit-learn's `feature_importances_` reports the **mean decrease in impurity (MDI)**: for each feature, it averages the total reduction in squared error (or Gini impurity) across every split in every tree that uses that feature.

In [ ]:
# Feature importance from the random forest
importances = pd.Series(rf.feature_importances_, index=feature_names)
importances_sorted = importances.sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2171b5' if v > importances.median() else '#aec7e8' for v in importances_sorted]
importances_sorted.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('Feature importance (mean decrease in impurity)')
ax.set_title('Random forest feature importance')
plt.tight_layout()
plt.show()

Compare this to a linear regression's coefficients. In a linear model, the coefficient on `bedrooms` tells you the marginal effect of one additional bedroom, holding all else equal — a causal-flavored interpretation (though only valid under strong assumptions). Feature importance tells you something different: how much the forest *uses* a feature for splitting, which reflects both direct effects and interactions. A feature can be important without having a simple linear relationship with the outcome.

:::{.callout-warning}
## MDI can be misleading
Mean decrease in impurity is biased toward features with many possible split points (high-cardinality features). A feature with 100 unique values gets more chances to be selected than a binary feature, even if both are equally predictive. **Permutation importance** — measuring how much test accuracy drops when a feature is randomly shuffled — is a more reliable alternative for comparing features on different scales.
:::


### How trees handle missing data

Unlike linear regression, which requires complete data for every row, decision trees can handle missing values directly. Different implementations use different strategies:

**Surrogate splits.** When a tree encounters a missing value at a split node, it looks for an alternative feature that produces a similar partition. If the best split is on `bedrooms` and a row has a missing bedroom count, the tree might split on `accommodates` instead (a correlated feature). The original CART algorithm uses this approach. (Note: scikit-learn's `DecisionTreeClassifier` does *not* implement surrogate splits — it requires complete data. Use `HistGradientBoostingClassifier` for native missing-value support.)

**Native missing-value handling.** XGBoost and LightGBM go further: during training, they learn which branch (left or right) missing values should follow at each split, choosing whichever direction reduces the loss more. No imputation needed — the algorithm treats "missing" as just another value to route.

**Why trees handle this well.** A decision tree asks a yes/no question at each node. Missing values simply need a rule for which branch to follow. Linear models, by contrast, need a numeric value to multiply by a coefficient — a missing value breaks the computation entirely.

:::{.callout-tip}
## Think About It
If 30% of your `bedrooms` column is missing and you impute with the median before fitting a tree, what information have you destroyed? Would the tree have been better off seeing the missingness directly?
:::

In practice, tree-based models (especially gradient-boosted trees like XGBoost) are the most popular choice for tabular data with missing values — precisely because they don't require the analyst to make imputation decisions that could introduce bias.


## Key takeaways

- **Decision trees find splits automatically — no manual feature engineering.** The model examines every feature and every threshold, choosing splits that minimize prediction error. No need for one-hot encoding, polynomial terms, or interaction terms.

- **A single deep tree overfits: perfect training score, poor test score.** With no depth limit, a tree memorizes the training data. The training R² approaches 1 while the test R² deteriorates.

- **Cross-validation picks the best hyperparameter.** Sweep candidate tree depths, score each by 5-fold CV on the training set, and pick the depth with the highest CV R². The same procedure selects polynomial degree, lasso α, and any other complexity knob.

- **Random forests average many overfit trees; the average is stable.** Each tree sees different bootstrap data and different feature subsets. The noise cancels; the signal persists.

- **More trees never hurts (unlike more polynomial features).** Test R² improves and then plateaus as we add trees — no U-shaped overfitting curve.

- **Trees handle missing data and categories natively.** Surrogate splits and native missing-value routing make imputation unnecessary. Categories can be split directly without one-hot encoding.

- **Feature importance ranks which variables the forest relies on.** Mean decrease in impurity (MDI) measures how much each feature contributes to reducing prediction error, but beware of bias toward high-cardinality features.

- **Classification trees use Gini impurity instead of squared error, but everything else stays the same.** Greedy splitting, depth control by CV, random forest ensembling — the entire framework transfers from regression to classification by swapping the splitting criterion.

- **Trees vs. logistic regression is a real design choice.** Logistic regression gives interpretable coefficients (log-odds). Trees capture nonlinear patterns without manual feature engineering. Random forests trade interpretability for consistently strong performance.


## Study guide

### Key ideas

- **Decision tree** — a model that recursively partitions the feature space by asking if/then questions about individual features, predicting the mean outcome (regression) or majority class (classification) in each leaf.
- **Recursive splitting** — the greedy algorithm that builds a tree: at each node, choose the feature and threshold that minimize prediction error in the child nodes.
- **Overfitting in trees** — a tree with no depth limit memorizes the training data (training R² ≈ 1) but performs poorly on new data. Controlling depth limits this memorization.
- **Bagging (bootstrap aggregating)** — training many models on different bootstrap samples of the data and averaging their predictions. Reduces variance without increasing bias.
- **Feature subsampling** — at each split, considering only a random subset of features. This decorrelates the trees in the ensemble, making the average more effective.
- **Random forest** — an ensemble of bagged decision trees with feature subsampling. More trees = better or flat, never worse.
- **Feature importance (MDI)** — mean decrease in impurity: the average reduction in squared error (or Gini impurity) across all splits using a given feature. Biased toward high-cardinality features; permutation importance is a more robust alternative.
- **Missing data in trees** — trees handle missing values natively via surrogate splits or learned missing-value routing, unlike linear models that require imputation.
- **Cross-validation for tree depth** — applying 5-fold CV to select the best `max_depth`, the same way we selected polynomial degree and lasso penalty in Chapter 6.
- **Gini impurity** — the classification splitting criterion $\hat{p}(1-\hat{p})$; measures how mixed a node is. Replaces squared error when the target is categorical.
- **Classification tree** — a decision tree that predicts the majority class in each leaf, using Gini impurity to choose splits. Same greedy recursive algorithm as regression trees.
- **Random forest classifier** — an ensemble of classification trees using bagging and feature subsampling. Uses majority vote (or averaged predicted probabilities) instead of averaging predictions.
- **Trees vs. logistic regression** — logistic regression gives interpretable coefficients (log-odds); trees capture nonlinear patterns without feature engineering. Random forests trade interpretability for strong default performance.

### Computational tools

- `DecisionTreeRegressor(max_depth=4).fit(X, y)` — fits a regression tree with controlled depth
- `DecisionTreeClassifier(max_depth=5).fit(X, y)` — fits a classification tree; uses Gini impurity
- `RandomForestRegressor(n_estimators=100).fit(X, y)` — fits a random forest; more trees = better or flat, never worse
- `RandomForestClassifier(n_estimators=100).fit(X, y)` — fits a classification forest; uses majority vote
- `plot_tree(tree, feature_names=..., filled=True)` — visualizes a fitted decision tree
- `export_text(tree, feature_names=...)` — prints the tree's decision rules as text
- `rf.feature_importances_` — array of feature importance scores (mean decrease in impurity)
- `confusion_matrix(y_true, y_pred)` — 2×2 matrix of true/false positives and negatives
- `roc_curve(y_true, y_prob)` — computes FPR/TPR pairs for plotting the ROC curve
- `roc_auc_score(y_true, y_prob)` — area under the ROC curve; higher = better discrimination
- `train_test_split(X, y, test_size=0.3)` — splits data into training and test sets for honest evaluation
- `cross_val_score(model, X, y, cv=5)` — 5-fold cross-validation; use to choose tree depth

### For the quiz

You are responsible for: how a decision tree splits (greedy recursive partitioning), why deep trees overfit, how bagging and feature subsampling combine to form a random forest, why adding more trees never hurts, feature importance interpretation, the advantages of trees over linear models for handling missing data and categorical features, how to use cross-validation to choose tree depth (same principle as choosing polynomial degree or lasso $\alpha$), classification trees and Gini impurity, confusion matrices and ROC curves for tree classifiers, and the tradeoffs between trees and logistic regression.